# Separate & Sequential Learning — Evaluation Report

Loads pre-trained single-task models, runs per-task evaluation and sequential
pipeline benchmark. Training was done separately (models in `models/`).

In [1]:
import sys, os, json, ast
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
SRC_DIR = os.path.join(PROJECT_ROOT, 'src')
for d in [SRC_DIR, os.path.join(SRC_DIR, 'jointlearning')]:
    if d not in sys.path:
        sys.path.insert(0, d)

import torch, pandas as pd
from tqdm import tqdm
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from jointlearning.model import JointCausalModel
from jointlearning.evaluate_joint_causal_model import evaluate_model
from jointlearning.dataset_collator import CausalDataset, CausalDatasetCollator
from jointlearning.utility import set_seed, seed_worker
from evaluate_sequential import sequential_predict, sequential_to_doccano
from analysis.causal_eval import evaluate, display_results
from cmp_config import *

set_seed(SEED)
print(f'Device: {DEVICE}  |  Seed: {SEED}')

Device: cuda  |  Seed: 8642


In [2]:
# Verify all model checkpoints exist
for task in ['cls', 'bio', 'rel']:
    p = MODEL_PATHS[task]
    assert os.path.exists(p), f'Missing: {p}'
    print(f'{task.upper()}: {p}  ({os.path.getsize(p)/1e6:.0f} MB)')
print('\nAll models present.')

CLS: /home/rnorouzini/JointLearning/reviewer_extra_analysis/comparison_learning/models/cls/separate_cls.pt  (449 MB)
BIO: /home/rnorouzini/JointLearning/reviewer_extra_analysis/comparison_learning/models/bio/separate_bio.pt  (449 MB)
REL: /home/rnorouzini/JointLearning/reviewer_extra_analysis/comparison_learning/models/rel/separate_rel.pt  (449 MB)

All models present.


In [3]:
# Load training histories
histories = {}
for task in ['cls', 'bio', 'rel']:
    hist_path = os.path.join(os.path.dirname(MODEL_PATHS[task]), 'training_history.json')
    if os.path.exists(hist_path):
        with open(hist_path) as f:
            h = json.load(f)
        histories[task] = h
        best_f1 = max(h['val_overall_f1'])
        best_epoch = h['val_overall_f1'].index(best_f1) + 1
        print(f'{task.upper()}: best val F1={best_f1:.4f} (epoch {best_epoch}/{len(h["val_overall_f1"])})')
    else:
        print(f'{task.upper()}: no history file')

CLS: best val F1=0.8117 (epoch 9/19)
BIO: best val F1=0.4916 (epoch 19/20)
REL: best val F1=0.8687 (epoch 7/17)


In [4]:
# Show task weights used
print('Task loss weights during training:')
for task, w in TASK_WEIGHTS.items():
    print(f'  {task.upper()}-only: {w}')

Task loss weights during training:
  CLS-only: {'cls': 1.0, 'bio': 0.0, 'rel': 0.0}
  BIO-only: {'cls': 0.0, 'bio': 1.0, 'rel': 0.0}
  REL-only: {'cls': 0.0, 'bio': 0.0, 'rel': 1.0}


---
## Section 1: Per-task evaluation of separate models (test set)

In [5]:
# Build test DataLoader
test_df = pd.read_csv(TEST_DATA_PATH)
test_ds = CausalDataset(test_df, tokenizer_name='bert-base-uncased', max_length=512)
test_coll = CausalDatasetCollator(tokenizer=test_ds.tokenizer)
test_ldr = DataLoader(test_ds, batch_size=16, collate_fn=test_coll, shuffle=False, worker_init_fn=seed_worker)
print(f'Test: {len(test_df)} sentences, {len(test_ldr)} batches')

Test: 452 sentences, 29 batches


In [6]:
def load_model(ckpt_path):
    m = JointCausalModel(**MODEL_CONFIG)
    m.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    m.to(DEVICE).eval()
    return m

separate_results = {}
task_keys = {'cls': 'task_cls', 'bio': 'task_bio', 'rel': 'task_relation'}
task_names = {'cls': 'Classification', 'bio': 'BIO Tagging', 'rel': 'Relation Extraction'}

for task in ['cls', 'bio', 'rel']:
    print(f'\n{"="*60}')
    print(f'  {task.upper()}-only model')
    print(f'{"="*60}')
    m = load_model(MODEL_PATHS[task])
    r = evaluate_model(m, test_ldr, DEVICE, id2label_cls, id2label_bio, id2label_rel)
    separate_results[task] = r
    
    tk = task_keys[task]
    tm = r.get(tk, {})
    if isinstance(tm, dict) and 'macro avg' in tm:
        ma = tm['macro avg']
        print(f'  Macro F1: {ma["f1-score"]:.4f}  P: {ma["precision"]:.4f}  R: {ma["recall"]:.4f}')
        # Also show per-class
        for k, v in tm.items():
            if isinstance(v, dict) and 'f1-score' in v:
                print(f'    {k}: F1={v["f1-score"]:.4f} P={v["precision"]:.4f} R={v["recall"]:.4f} S={v["support"]}')
    
    # Show inactive tasks are near zero
    for other_tk, other_name in [('task_cls','CLS'), ('task_bio','BIO'), ('task_relation','REL')]:
        if other_tk != tk:
            ot = r.get(other_tk, {})
            if isinstance(ot, dict) and 'macro avg' in ot:
                print(f'    (inactive {other_name} F1: {ot["macro avg"]["f1-score"]:.4f})')
    del m
    torch.cuda.empty_cache()


  CLS-only model


MRO for JointCausalModel: (<class 'jointlearning.model.JointCausalModel'>, <class 'torch.nn.modules.module.Module'>, <class 'huggingface_hub.hub_mixin.PyTorchModelHubMixin'>, <class 'huggingface_hub.hub_mixin.ModelHubMixin'>, <class 'object'>)


Evaluating:   0%|          | 0/29 [00:00<?, ?it/s]

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Evaluating:   3%|▎         | 1/29 [00:00<00:06,  4.02it/s]

...

Evaluating:  55%|█████▌    | 16/29 [00:00<00:00, 35.13it/s]

Evaluating:  69%|██████▉   | 20/29 [00:00<00:00, 36.29it/s]

Evaluating:  83%|████████▎ | 24/29 [00:00<00:00, 36.29it/s]

Evaluating:  97%|█████████▋| 28/29 [00:00<00:00, 35.40it/s]

  Macro F1: 0.8353  P: 0.8283  R: 0.8465
    Rel_None: F1=0.8801 P=0.9130 R=0.8496 S=605.0
    Rel_CE: F1=0.7904 P=0.7437 R=0.8435 S=313.0
    macro avg: F1=0.8353 P=0.8283 R=0.8465 S=918.0
    weighted avg: F1=0.8495 P=0.8552 R=0.8475 S=918.0
    (inactive CLS F1: 0.3382)
    (inactive BIO F1: 0.0215)


In [7]:
# Table 1
rows = []
for task in ['cls', 'bio', 'rel']:
    r = separate_results[task]
    tk = task_keys[task]
    tm = r.get(tk, {})
    if isinstance(tm, dict) and 'macro avg' in tm:
        ma = tm['macro avg']
        rows.append({'Model': f'{task.upper()}-only', 'Task': task_names[task],
                     'Macro F1': round(ma['f1-score'], 4),
                     'Precision': round(ma['precision'], 4),
                     'Recall': round(ma['recall'], 4)})

t1 = pd.DataFrame(rows)
print('Table 1: Per-task evaluation of separate models (test set)')
print(t1.to_string(index=False))

Table 1: Per-task evaluation of separate models (test set)
   Model                Task  Macro F1  Precision  Recall
CLS-only      Classification    0.8295     0.8326  0.8307
BIO-only         BIO Tagging    0.4897     0.4807  0.5327
REL-only Relation Extraction    0.8353     0.8283  0.8465


---
## Section 2: Sequential pipeline evaluation

CLS → BIO → REL, CLS decision is final. Tokenization strips `;;` to match training.

In [8]:
# Load models for pipeline
cls_m = load_model(MODEL_PATHS['cls'])
bio_m = load_model(MODEL_PATHS['bio'])
rel_m = load_model(MODEL_PATHS['rel'])
tok = AutoTokenizer.from_pretrained(MODEL_CONFIG['encoder_name'])
texts = test_df['text'].tolist()

# Run sequential prediction
all_preds = []
BS = 32
for bi in tqdm(range((len(texts) + BS - 1)//BS), desc='Sequential predict'):
    batch = texts[bi*BS:(bi+1)*BS]
    all_preds.extend(sequential_predict(cls_m, bio_m, rel_m, batch, tok, DEVICE))

n_c = sum(1 for r in all_preds if r['causal'])
n_s = sum(len(r.get('spans',[])) for r in all_preds)
n_r = sum(len(r.get('relations',[])) for r in all_preds)
print(f'Predicted: {n_c}/{len(all_preds)} causal, {n_s} spans, {n_r} relations')

MRO for JointCausalModel: (<class 'jointlearning.model.JointCausalModel'>, <class 'torch.nn.modules.module.Module'>, <class 'huggingface_hub.hub_mixin.PyTorchModelHubMixin'>, <class 'huggingface_hub.hub_mixin.ModelHubMixin'>, <class 'object'>)


MRO for JointCausalModel: (<class 'jointlearning.model.JointCausalModel'>, <class 'torch.nn.modules.module.Module'>, <class 'huggingface_hub.hub_mixin.PyTorchModelHubMixin'>, <class 'huggingface_hub.hub_mixin.ModelHubMixin'>, <class 'object'>)


MRO for JointCausalModel: (<class 'jointlearning.model.JointCausalModel'>, <class 'torch.nn.modules.module.Module'>, <class 'huggingface_hub.hub_mixin.PyTorchModelHubMixin'>, <class 'huggingface_hub.hub_mixin.ModelHubMixin'>, <class 'object'>)


Sequential predict:   0%|          | 0/15 [00:00<?, ?it/s]

Sequential predict:   7%|▋         | 1/15 [00:00<00:02,  4.94it/s]

...

Sequential predict:  80%|████████  | 12/15 [00:02<00:00,  4.91it/s]

Sequential predict:  87%|████████▋ | 13/15 [00:02<00:00,  4.78it/s]

Sequential predict:  93%|█████████▎| 14/15 [00:02<00:00,  4.53it/s]

Sequential predict: 100%|██████████| 15/15 [00:02<00:00,  5.17it/s]

Predicted: 244/452 causal, 725 spans, 517 relations


In [9]:
# Convert to Doccano & evaluate
pred_df = sequential_to_doccano(all_preds)
csv_path = os.path.join(PREDICTIONS_DIR, 'sequential_predictions_doccano.csv')
pred_df.to_csv(csv_path, index=False)

gold_df = pd.read_csv(TEST_DATA_PATH)
scenarios = ['all_documents', 'filtered_causal']
eval_modes = ['discovery', 'coverage']

print('Table 2: Sequential pipeline end-to-end results\n' + '='*70)
t2_rows = []
for scenario in scenarios:
    for eval_mode in eval_modes:
        r = evaluate(gold_df, pred_df, scenario=scenario, eval_mode=eval_mode)
        display_results(r, title_prefix=f'Sequential | {scenario} | {eval_mode}')
        t2_rows.append({
            'Scenario': scenario, 'Eval Mode': eval_mode,
            'Task1 F1': round(r['Task1']['F1'], 4),
            'Task2 Macro': round(r['Task2_macro']['F1'], 4),
            'Task3 F1': round(r['Task3']['F1'], 4),
            'Total Macro': round(r['Total_Macro']['F1'], 4),
        })

t2 = pd.DataFrame(t2_rows)
print(t2.to_string(index=False))
t2.to_csv(os.path.join(PREDICTIONS_DIR, 'sequential_metrics.csv'), index=False)


SEQUENTIAL → DOCCANO CONVERSION COMPLETE
Total samples: 452
Causal: 244  Non-causal: 208
Total entities: 983
Total relations: 517
Table 2: Sequential pipeline end-to-end results

            Sequential | all_documents | discovery Results            

--- Task1 ---
  TP          :      192
  FP          :       45
  FN          :       29
  TN          :      186
  Precision   :   0.8101
  Recall      :   0.8688
  F1          :   0.8384
  Accuracy    :   0.8363
  N           :      452

--- Task2 ---
  Label: cause
    Precision   :   0.5781
    Recall      :   0.7088
    F1          :   0.6368
    TP          :      185
    FP          :      135
    FN          :       76
  Label: effect
    Precision   :   0.6188
    Recall      :   0.7778
    F1          :   0.6892
    TP          :      224
    FP          :      138
    FN          :       64

--- Task2_macro ---
  Precision   :   0.5985
  Recall      :   0.7433
  F1          :   0.6630
  TP          :      409
  FP          :   


           Sequential | filtered_causal | discovery Results           

--- Task1 ---
  TP          :      192
  FP          :       45
  FN          :       29
  TN          :      186
  Precision   :   0.8101
  Recall      :   0.8688
  F1          :   0.8384
  Accuracy    :   0.8363
  N           :      452

--- Task2 ---
  Label: cause
    Precision   :   0.7061
    Recall      :   0.8043
    F1          :   0.7520
    TP          :      185
    FP          :       77
    FN          :       45
  Label: effect
    Precision   :   0.7417
    Recall      :   0.8716
    F1          :   0.8014
    TP          :      224
    FP          :       78
    FN          :       33

--- Task2_macro ---
  Precision   :   0.7239
  Recall      :   0.8380
  F1          :   0.7767
  TP          :      409
  FP          :      155
  FN          :       78

--- Task3 ---
  TP          :       55
  FP          :      109
  FN          :      203
  Accuracy    :   0.1499
  Precision   :   0.3354
  Recal

---
## Section 3: Joint vs. Sequential comparison

Both evaluated via `causal_eval.evaluate()` on Doccano output (apples-to-apples).
Joint numbers from `Notebooks/evaluation_report.md` (bert-softmax, threshold=0.5, cls+span).

In [10]:
joint = pd.DataFrame([
    {'Model':'Joint','Scenario':'all_documents','Eval Mode':'coverage',
     'Task1 F1':0.7981,'Task2 Macro':0.7136,'Task3 F1':0.5924,'Total Macro':0.7014},
    {'Model':'Joint','Scenario':'all_documents','Eval Mode':'discovery',
     'Task1 F1':0.7981,'Task2 Macro':0.6904,'Task3 F1':0.5433,'Total Macro':0.6773},
    {'Model':'Joint','Scenario':'filtered_causal','Eval Mode':'coverage',
     'Task1 F1':0.7981,'Task2 Macro':0.8556,'Task3 F1':0.6941,'Total Macro':0.7826},
    {'Model':'Joint','Scenario':'filtered_causal','Eval Mode':'discovery',
     'Task1 F1':0.7981,'Task2 Macro':0.8361,'Task3 F1':0.6468,'Total Macro':0.7603},
])

seq = t2.copy()
seq.insert(0, 'Model', 'Sequential')

t3 = pd.concat([joint, seq], ignore_index=True)

print('Table 3: Joint vs Sequential')
print('='*80)
print(t3.to_string(index=False))

print('\nGap (Joint - Sequential Total Macro):')
for (_,j),(_,s) in zip(joint.iterrows(), seq.iterrows()):
    gap = j['Total Macro'] - s['Total Macro']
    print(f'  {j["Scenario"]:20s} {j["Eval Mode"]:10s}  gap = {gap:+.4f}')

Table 3: Joint vs Sequential
     Model        Scenario Eval Mode  Task1 F1  Task2 Macro  Task3 F1  Total Macro
     Joint   all_documents  coverage    0.7981       0.7136    0.5924       0.7014
     Joint   all_documents discovery    0.7981       0.6904    0.5433       0.6773
     Joint filtered_causal  coverage    0.7981       0.8556    0.6941       0.7826
     Joint filtered_causal discovery    0.7981       0.8361    0.6468       0.7603
Sequential   all_documents discovery    0.8384       0.6630    0.2273       0.5762
Sequential   all_documents  coverage    0.8384       0.6988    0.2560       0.5978
Sequential filtered_causal discovery    0.8384       0.7767    0.2607       0.6253
Sequential filtered_causal  coverage    0.8384       0.8091    0.2922       0.6466

Gap (Joint - Sequential Total Macro):
  all_documents        coverage    gap = +0.1252
  all_documents        discovery   gap = +0.0795
  filtered_causal      coverage    gap = +0.1573
  filtered_causal      discovery   gap

In [11]:
# Cleanup
for m in [cls_m, bio_m, rel_m]:
    del m
torch.cuda.empty_cache()
print('Done.')

Done.
